In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 09 · Code-evaluation drills: spot the bug

**Primer section:** §11.2 code-evaluation drills.

Eight short snippets, each with one deliberate security flaw from the primer's list: token
passthrough, `aud` not checked, allow-unknown-tools, a confirmation UI that shows the model's summary,
a secret in `tool_context.state`, substring egress matching, a DPoP verifier without `jti` tracking,
and a `principalSet` matcher that prefix-matches. Each cell shows the **buggy** snippet, the **fixed**
version, and a runnable demonstration that proves the difference with the library. Read the buggy
version first and name the bug before scrolling.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

import json

import jwt

from agentsec.agents import REFUNDS, LocalStack, Step, reset_demo_state
from agentsec.audit import AuditLog
from agentsec.config import Settings
from agentsec.guardrails import EgressPolicy
from agentsec.identity import (
    AgentIdentity,
    AuthorityContext,
    DPoP,
    InsufficientScope,
    InvalidAudience,
    PrincipalSet,
    ReplayDetected,
    TokenIssuer,
    UserPrincipal,
    member_matches,
    public_jwk,
)
from agentsec.policy import Decision, Effect, Policy, PolicyEngine, ToolCallRequest
from agentsec.runtime import run_turn, seed_session
from agentsec.secrets import SecretValue, redact

reset_demo_state()

settings = Settings()
agent = settings.agent_identity()
issuer = TokenIssuer()
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")
policy = Policy.from_yaml(settings.policy_path)

MCP_URL = "https://tickets.example/mcp"
PAYMENTS = "https://payments.example"
TD = agent.trust_domain

def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

## Drill 1 — token passthrough (MCP §7.1, ASI07)

An MCP server refunds through an upstream payments API. Review `refund_upstream_buggy`: what does it
put on the wire to the payments API, and why is that a confused deputy?

In [ ]:
# BUG: the server forwards the token it *received* (aud = this MCP server) to the payments API.
def refund_upstream_buggy(inbound_token: str) -> dict:
    return {"Authorization": f"Bearer {inbound_token}"}

# FIX: the MCP server is its own OAuth client to payments — a separate token for THAT audience,
# under the server's own identity, with the user recorded as on_behalf_of. The inbound token never leaves.
def refund_upstream_fixed(inbound_claims, *, actor: str) -> dict:
    upstream = issuer.mint(subject=actor, audience=PAYMENTS, scope="payments:refund", ttl=60,
                           extra={"on_behalf_of": inbound_claims.subject})
    return {"Authorization": f"Bearer {upstream}"}

# --- demonstration ---
inbound = issuer.mint(subject="u-ana", audience=MCP_URL, scope="tickets:write", extra={"act": {"sub": agent.spiffe_id}})
inbound_claims = issuer.verify(inbound, audience=MCP_URL)

def payments_api_accepts(headers: dict) -> bool:   # a *correct* payments API validates audience + scope
    token = headers["Authorization"].split()[1]
    try:
        issuer.verify(token, audience=PAYMENTS, required_scopes={"payments:refund"})
        return True
    except (InvalidAudience, InsufficientScope):
        return False

assert not payments_api_accepts(refund_upstream_buggy(inbound))            # passthrough: wrong audience
fixed_headers = refund_upstream_fixed(inbound_claims, actor=agent.spiffe_id)
assert payments_api_accepts(fixed_headers)
assert peek(fixed_headers["Authorization"].split()[1])["aud"] == PAYMENTS != MCP_URL
print("passthrough rejected by a correct upstream; exchanged token accepted — and a *lax* upstream would have made the passthrough a confused deputy")

## Drill 2 — signature checked, audience not (§3.5, §7.1)

A resource server's verifier. It checks the signature, issuer and expiry. What is missing, and what
can an attacker do with a token they legitimately obtained for a *different* API?

In [ ]:
# BUG: verify_aud=False — any token signed by our AS, for any API, is accepted here.
def verify_buggy(token: str) -> dict:
    return jwt.decode(token, issuer.public_key_pem, algorithms=["RS256"], issuer=issuer.issuer, options={"verify_aud": False})

# FIX: the resource server verifies that the token was issued FOR IT (aud == its canonical URI).
def verify_fixed(token: str):
    return issuer.verify(token, audience=MCP_URL)

# --- demonstration ---
for_calendar = issuer.mint(subject="u-ana", audience="https://calendar.example", scope="calendar:read")
assert verify_buggy(for_calendar)["sub"] == "u-ana"        # accepted although it was minted for the calendar API
try:
    verify_fixed(for_calendar)
    raise AssertionError("must reject")
except InvalidAudience as e:
    print("fixed verifier:", type(e).__name__, "-", e)
assert verify_fixed(issuer.mint(subject="u-ana", audience=MCP_URL, scope="tickets:read")).subject == "u-ana"

## Drill 3 — the engine that allows unknown tools (§4.2)

A "pragmatic" policy engine: tools that nobody classified are assumed harmless. Where does `run_sql`
end up, and what is the one-line fix?

In [ ]:
# BUG: unknown tool → ALLOW ("assume harmless"). The model can call anything not yet in the policy file.
class BuggyEngine(PolicyEngine):
    def evaluate(self, req: ToolCallRequest) -> Decision:
        if req.tool not in self.policy.tools:
            return Decision(Effect.ALLOW, ["tool not in policy — assuming harmless"])
        return super().evaluate(req)

# FIX: unknown tool → DENY. Deny-by-default is the whole point; the library's engine already does this.
class FixedEngine(PolicyEngine):
    def evaluate(self, req: ToolCallRequest) -> Decision:
        if req.tool not in self.policy.tools:
            return Decision(Effect.DENY, [f"tool {req.tool!r} is not in policy (default deny)"])
        return super().evaluate(req)

# --- demonstration ---
delegated = AuthorityContext.delegated(agent, ana, {"customers:read", "payments:refund"})
req = ToolCallRequest(tool="run_sql", args={"query": "drop table customers"}, authority=delegated)
assert BuggyEngine(policy).evaluate(req).effect is Effect.ALLOW
assert FixedEngine(policy).evaluate(req).effect is Effect.DENY
assert PolicyEngine(policy).evaluate(req).effect is Effect.DENY
try:
    Policy.from_dict({"default": "allow", "tools": {}})
    raise AssertionError("must reject")
except ValueError:
    print("run_sql: buggy=ALLOW fixed=DENY; and `default: allow` is not even a valid policy")

## Drill 4 — the confirmation UI shows the model's summary (§4.4, ASI09)

The front-end renders a confirmation dialog. It shows what the *model said* it will do. A hijacked
model calls `issue_refund(O-5003, 500 USD)` while telling the user it is refunding 35 USD on their own
order. What must the dialog render instead?

In [ ]:
# BUG: renders the model's prose. The human approves a sentence, not the call that will execute.
def render_confirmation_buggy(model_summary: str, pending) -> str:
    return f"Assistant: {model_summary}  [Approve] [Reject]"

# FIX: render the actual pending call — tool, exact arguments, authority (for whom, by which agent), tier, policy hint.
def render_confirmation_fixed(model_summary: str, pending) -> str:
    call, who = pending.original_call, pending.payload["authority"]
    return (f"Approve {call['name']}({json.dumps(call['args'])}) on behalf of {who['user']} "
            f"by {who['agent'].rsplit('/', 1)[-1]} [tier={pending.payload['tier']}] — {pending.hint}  [Approve] [Reject]")

# --- demonstration ---
stack = LocalStack.create(Settings(), audit=AuditLog())
await seed_session(stack.runner, user_id="u-ana", session_id="d4", user={"subject": "u-ana", "email": "ana@customer.example"},
                   scopes=["customers:read", "orders:read", "payments:refund"])
stack.script(Step.call("issue_refund", order_id="O-5003", amount=500.0, currency="USD", reason="as instructed"), Step.say("ok"))
r = await run_turn(stack.runner, user_id="u-ana", session_id="d4", message="refund my museum pass")
pending = r.pending_confirmations[0]
model_summary = "I'll refund the 35 USD museum pass on your own order, as you asked."   # what a hijacked model *says*

buggy_ui = render_confirmation_buggy(model_summary, pending)
fixed_ui = render_confirmation_fixed(model_summary, pending)
print("buggy:", buggy_ui)
print("fixed:", fixed_ui)
assert "500" not in buggy_ui and "O-5003" not in buggy_ui
assert "O-5003" in fixed_ui and "500" in fixed_ui and "ana@customer.example" in fixed_ui and "destructive" in fixed_ui
assert REFUNDS == []

## Drill 5 — a secret parked in `tool_context.state` (§5)

A tool caches the CRM API key in session state "so the next call is faster". Session state is
persisted, serialised, and readable by anything that can read the session — including the model via
a prompt that asks for it. What should the tool do instead?

In [ ]:
class FakeToolContext:            # stands in for ADK's ToolContext: `.state` is session state
    def __init__(self):
        self.state: dict = {}

# BUG: raw secret in session state (persisted + serialisable + reachable from the context window).
def crm_lookup_buggy(tool_context, email: str, api_key: SecretValue) -> dict:
    tool_context.state["crm_api_key"] = api_key.reveal()
    headers = {"X-API-Key": tool_context.state["crm_api_key"]}
    return {"content": f"record for {email}", "sent_headers": sorted(headers)}

# FIX: reveal at the single point of use (the outbound request); keep only non-secret facts in state.
# In production the key is not even a parameter: Auth Manager / the broker hands it to ADK per call.
def crm_lookup_fixed(tool_context, email: str, api_key: SecretValue) -> dict:
    headers = {"X-API-Key": api_key.reveal()}
    tool_context.state["crm_last_lookup"] = email
    return {"content": f"record for {email}", "sent_headers": sorted(headers)}

# --- demonstration ---
api_key = SecretValue("k-123-crm-secret", name="crm-api-key")
buggy_ctx, fixed_ctx = FakeToolContext(), FakeToolContext()
crm_lookup_buggy(buggy_ctx, "ana@customer.example", api_key)
crm_lookup_fixed(fixed_ctx, "ana@customer.example", api_key)

serialised_buggy = json.dumps(buggy_ctx.state)        # what the session store / a session dump would contain
serialised_fixed = json.dumps(fixed_ctx.state)
print("buggy state:", serialised_buggy)
print("fixed state:", serialised_fixed)
print("redaction is only a backstop — it does not know this key name:", redact(buggy_ctx.state))
assert "k-123-crm-secret" in serialised_buggy
assert "k-123-crm-secret" not in serialised_fixed and "k-123-crm-secret" not in repr(api_key)

## Drill 6 — the egress check that matches a substring (§6.1)

`fetch_url` may only reach `docs.acme.example`. Review the allowlist check: list three URLs that pass
it and reach somewhere else.

In [ ]:
ALLOWED_HOSTS = {"docs.acme.example"}

# BUG: substring match on the whole URL string — attacker-controlled hosts, userinfo tricks and query strings all pass.
def egress_ok_buggy(url: str) -> bool:
    return any(host in url for host in ALLOWED_HOSTS)

# FIX: parse the URL, compare the *host* exactly (or by registered suffix), require https, block private ranges.
egress = EgressPolicy(allowed_hosts=ALLOWED_HOSTS)

def egress_ok_fixed(url: str) -> bool:
    return egress.check(url).allowed

# --- demonstration ---
tricks = [
    "https://docs.acme.example.evil.example/",          # attacker-owned subdomain
    "https://evil.example/?next=docs.acme.example",     # allowed host in the query string
    "https://docs.acme.example@evil.example/",          # userinfo trick: host is evil.example
    "http://docs.acme.example/",                        # plaintext
]
assert all(egress_ok_buggy(t) for t in tricks)
assert not any(egress_ok_fixed(t) for t in tricks)
assert egress_ok_fixed("https://docs.acme.example/refunds")
for t in tricks:
    print(f"buggy={egress_ok_buggy(t)!s:<5} fixed={egress_ok_fixed(t)!s:<5} {t}  ({egress.check(t).reason})")

## Drill 7 — the DPoP verifier that forgets `jti` (§3.5)

A gateway verifies DPoP proofs: signature, `htm`, `htu`, `ath`, freshness. What attack still works?

In [ ]:
# BUG: no jti replay cache — a captured (token, proof) pair can be replayed for as long as `iat` is fresh.
class DPoPVerifierBuggy:
    def verify(self, proof: str, *, method: str, url: str, token: str) -> dict:
        return DPoP.verify(proof, method=method, url=url, access_token=token)

# FIX: remember every jti seen (bounded by max_age in production) and reject a second presentation.
class DPoPVerifierFixed:
    def __init__(self):
        self.seen_jti: set[str] = set()

    def verify(self, proof: str, *, method: str, url: str, token: str) -> dict:
        return DPoP.verify(proof, method=method, url=url, access_token=token, seen_jti=self.seen_jti)

# --- demonstration ---
key = DPoP.generate_key()
token = issuer.mint_dpop_bound_token(subject="u-ana", audience=MCP_URL, dpop_public_jwk=public_jwk(key), scope="tickets:read")
proof = DPoP.proof(key, method="POST", url=MCP_URL, access_token=token)   # captured on the wire

buggy, fixed = DPoPVerifierBuggy(), DPoPVerifierFixed()
buggy.verify(proof, method="POST", url=MCP_URL, token=token)
buggy.verify(proof, method="POST", url=MCP_URL, token=token)          # replay accepted (!)
fixed.verify(proof, method="POST", url=MCP_URL, token=token)
try:
    fixed.verify(proof, method="POST", url=MCP_URL, token=token)
    raise AssertionError("replay must fail")
except ReplayDetected as e:
    print("fixed verifier:", e)

## Drill 8 — the `principalSet` matcher that prefix-matches (§3.3)

Fleet policy binds `principalSet://…/attribute.platformContainer/aiplatform/projects/98765432109`
(note: one digit short — a typo, or a neighbouring project). Which agents does the buggy matcher
select?

In [ ]:
# BUG: startswith on the attribute value — project 987654321098 matches a set written for 98765432109.
def pset_matches_buggy(member: str, who: AgentIdentity) -> bool:
    ps = PrincipalSet.parse(member)
    return who.trust_domain == ps.trust_domain and (who.platform_container or "").startswith(ps.value)

# FIX: compare whole segments — the attribute value must equal the agent's platformContainer exactly.
def pset_matches_fixed(member: str, who: AgentIdentity) -> bool:
    ps = PrincipalSet.parse(member)
    return who.trust_domain == ps.trust_domain and who.platform_container == ps.value

# --- demonstration ---
typo_set = f"principalSet://{TD}/attribute.platformContainer/aiplatform/projects/98765432109"
exact_set = f"principalSet://{TD}/attribute.platformContainer/aiplatform/projects/987654321098"
neighbour = AgentIdentity.for_agent_engine(project_number="987654321091", location="us-central1", engine_id="x", org_id=settings.org_id)

assert pset_matches_buggy(typo_set, agent) and pset_matches_buggy(typo_set, neighbour)   # both selected by a one-digit-short set
assert not pset_matches_fixed(typo_set, agent) and not pset_matches_fixed(typo_set, neighbour)
assert pset_matches_fixed(exact_set, agent) and member_matches(exact_set, agent) and not member_matches(typo_set, agent)
print("typo set selects with buggy matcher:", pset_matches_buggy(typo_set, agent), "| with fixed matcher:", pset_matches_fixed(typo_set, agent))

## How to run these as drills

For each snippet say, in order: *what it does*, *what it omits*, *what an attacker gains*, *the fix*,
and *which GCP control would have caught it anyway* (audience validation at the resource server and
gateway; deny-by-default in the runtime plugin and IAM; Agent Gateway's DPoP enforcement; Auth Manager
so tools never hold raw secrets; VPC-SC egress rules; exact-segment `principalSet` matching in IAM).

**In one sentence:** "Most agent security bugs are one missing check: the audience, the
allowlist default, the replay cache, the parsed host, the exact segment. I look for the check that
turns 'signed' into 'issued for me', 'not classified' into 'denied', and 'the model said' into 'the
call that will execute'."